In [17]:
# Standardbibliothek Panda für Tabellen in Python
import pandas as pd

df = pd.read_csv("Bundesliga1_2526.csv")
# df.head() zeigt mir nur die ersten 5 Einträge an
df_Saison2526.head()

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
0,D1,22/08/2025,19:30,Bayern Munich,RB Leipzig,6,0,H,3,0,...,1.98,1.88,1.98,1.93,1.99,1.93,1.90,1.86,2.07,1.92
1,D1,23/08/2025,14:30,Ein Frankfurt,Werder Bremen,4,1,H,2,0,...,1.83,2.03,2.02,1.91,1.91,2.03,1.83,1.93,1.91,2.06
2,D1,23/08/2025,14:30,Freiburg,Augsburg,1,3,A,0,3,...,1.93,1.93,1.97,1.95,1.97,1.93,1.90,1.86,2.03,1.96
3,D1,23/08/2025,14:30,Heidenheim,Wolfsburg,1,3,A,1,1,...,2.03,1.83,2.06,1.87,2.03,1.85,1.97,1.81,2.12,1.88
4,D1,23/08/2025,14:30,Leverkusen,Hoffenheim,1,2,A,1,1,...,1.98,1.88,1.97,1.95,1.98,2.02,1.88,1.86,2.00,1.97


In der Tabelle oben ist jede Zeile ein Spiel

In [18]:
df.shape
# df.shape gibt die Dimension des Data Frames an, damit kann man überprüfen, wie viel Zeilen und Spalten das Data Frame hat. --> Eine Bundesliga Saison hat 306 Spiele

(306, 131)

In [19]:
# Listet alle Spaltennamen auf. Wir brauchen nur "Date, HomeTeam, AwayTeam, FTHG, FTAG, FTR, HS, AS, HST, AST, HY, AY, HR, AR" - "Spieltag, HeimTeam, Gastteam, Tore Heim, Tore Gast am Spielende, Ergebnis: H = Heimsieg, D = Remis, A = Auswärtssieg, Schüsse Heim, Schüsse Gast, Schüsse aufs Tor Heim/Gast, Gelbe und Rote Karte Heim/Gast"
df.columns

Index(['Div', 'Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR',
       'HTHG', 'HTAG',
       ...
       'B365CAHH', 'B365CAHA', 'PCAHH', 'PCAHA', 'MaxCAHH', 'MaxCAHA',
       'AvgCAHH', 'AvgCAHA', 'BFECAHH', 'BFECAHA'],
      dtype='object', length=131)

In [20]:
df["FTR"].value_counts()
# Zählt wie oft H,D und A vorkommen. --> Erste Statistik kann man hier ablesen

FTR
H    134
A     97
D     75
Name: count, dtype: int64

In [21]:
df[["FTHG", "FTAG"]].describe()
# Zeigt Mittelwert, Minimum, Maximum für Tore an

,FTHG,FTAG
count,306.000000,306.000000
mean,1.777778,1.457516
std,1.398933,1.264843
min,0.000000,0.000000
25%,1.000000,1.000000
50%,2.000000,1.000000
75%,3.000000,2.000000
max,8.000000,6.000000


In [22]:
df_Saison2122 = pd.read_csv("Bundesliga1_2122.csv")
df_Saison2122["FTR"].value_counts()
# Zählt wie oft H,D und A vorkommen, um diese mit der letzten Saison zu vergleichen

FTR
H    143
A     90
D     73
Name: count, dtype: int64

In [23]:
print("2025/26:")
print(df["FTR"].value_counts(normalize=True).round(3) * 100)

print("2020/21 (Geistersaison):")
print(df_Saison2122["FTR"].value_counts(normalize=True).round(3) * 100)
# direkter Vergleich der Ergebnisse (H: Home, A: Away, D: Draw) zweier Saisons

2025/26:
FTR
H    43.8
A    31.7
D    24.5
Name: proportion, dtype: float64
2020/21 (Geistersaison):
FTR
H    46.7
A    29.4
D    23.9
Name: proportion, dtype: float64


In [24]:
# Daten sollen nun bereinigt und zusammengeführt werden, aus 8 Roh-Dateien werden drei saubere Tabellen
from pathlib import Path

spalten = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR",
           "HS", "AS", "HST", "AST", "HY", "AY", "HR", "AR"]

teile = []
for datei in sorted(Path("daten/roh").glob("D1_*.csv")):
    code = datei.stem.split("_")[1]          # z. B. "2425"
    saison = f"20{code[:2]}/{code[2:]}"      # wird zu "2024/25"
    d = pd.read_csv(datei, usecols=spalten)
    d["Date"] = pd.to_datetime(d["Date"], dayfirst=True)
    d["Saison"] = saison
    teile.append(d)

spiele = pd.concat(teile, ignore_index=True)
spiele = spiele.dropna(subset=["FTHG", "FTAG"])
spiele["Saison"].value_counts().sort_index()

Saison
2018/19    306
2019/20    306
2020/21    306
2021/22    306
2022/23    306
2023/24    306
2024/25    306
2025/26    306
Name: count, dtype: int64

In [25]:
# Die Englischen Tabellennamen auf Deutsch ändern
spiele = spiele.rename(columns={
    "Date": "Datum", "HomeTeam": "Heim", "AwayTeam": "Gast",
    "FTHG": "ToreHeim", "FTAG": "ToreGast", "FTR": "Ergebnis",
    "HS": "SchuesseHeim", "AS": "SchuesseGast",
    "HST": "AufsTorHeim", "AST": "AufsTorGast",
    "HY": "GelbHeim", "AY": "GelbGast", "HR": "RotHeim", "AR": "RotGast",
})
spiele["SpielID"] = range(1, len(spiele) + 1)

In [26]:
heim = pd.DataFrame({
    "SpielID": spiele["SpielID"], "Saison": spiele["Saison"], "Datum": spiele["Datum"],
    "Team": spiele["Heim"], "Gegner": spiele["Gast"], "Ort": "Heim",
    "Tore": spiele["ToreHeim"], "Gegentore": spiele["ToreGast"],
})
gast = pd.DataFrame({
    "SpielID": spiele["SpielID"], "Saison": spiele["Saison"], "Datum": spiele["Datum"],
    "Team": spiele["Gast"], "Gegner": spiele["Heim"], "Ort": "Auswärts",
    "Tore": spiele["ToreGast"], "Gegentore": spiele["ToreHeim"],
})
team_spiele = pd.concat([heim, gast], ignore_index=True)

def punkte(zeile):
    if zeile["Tore"] > zeile["Gegentore"]:
        return 3
    if zeile["Tore"] == zeile["Gegentore"]:
        return 1
    return 0

team_spiele["Punkte"] = team_spiele.apply(punkte, axis=1)
team_spiele["Resultat"] = team_spiele["Punkte"].map(
    {3: "Sieg", 1: "Unentschieden", 0: "Niederlage"})
team_spiele.head()

,SpielID,Saison,Datum,Team,Gegner,Ort,Tore,Gegentore,Punkte,Resultat
0,1,2018/19,2018-08-24,Bayern Munich,Hoffenheim,Heim,3,1,3,Sieg
1,2,2018/19,2018-08-25,Fortuna Dusseldorf,Augsburg,Heim,1,2,0,Niederlage
2,3,2018/19,2018-08-25,Freiburg,Ein Frankfurt,Heim,0,2,0,Niederlage
3,4,2018/19,2018-08-25,Hertha,Nurnberg,Heim,1,0,3,Sieg
4,5,2018/19,2018-08-25,M'gladbach,Leverkusen,Heim,2,0,3,Sieg


In [27]:
saisons = pd.DataFrame({"Saison": sorted(spiele["Saison"].unique())})

Path("daten/sauber").mkdir(parents=True, exist_ok=True)
spiele.to_csv("daten/sauber/spiele.csv", index=False)
team_spiele.to_csv("daten/sauber/team_spiele.csv", index=False)
saisons.to_csv("daten/sauber/saisons.csv", index=False)